# 01 · Llama inference from scratch

Llama-3.2-1B is a decoder-only transformer:

```
tokens                                  (B, T)
 └─ embedding E                         (B, T, d)
 └─ N × block
      h  = x + Attn(RMSNorm(x))         RoPE on q, k · grouped-query attention · KV cache
      x' = h + FFN(RMSNorm(h))          SwiGLU
 └─ RMSNorm
 └─ Eᵀ                                  (B, T, V)   logits
```

In [ ]:
import math
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Config

| Symbol | Value | Meaning |
|---|---|---|
| $d$ | 2048 | model width (residual stream) |
| $N$ | 16 | blocks |
| $H$ | 32 | query heads |
| $H_{kv}$ | 8 | key/value heads |
| $d_h = d/H$ | 64 | head dimension |
| $g = H/H_{kv}$ | 4 | query heads per K/V head |
| $d_{ff}$ | $8192 = 4d$ | FFN hidden width |
| $V$ | 128256 | vocabulary |
| $b$ | 500000 | RoPE base |

In [ ]:
@dataclass
class Config:
    vocab_size: int = 128256
    dim: int = 2048
    n_layers: int = 16
    n_heads: int = 32
    n_kv_heads: int = 8                    # 4 query heads share each K/V head
    ffn_dim: int = 8192
    norm_eps: float = 1e-5
    rope_theta: float = 500000.0           # b
    rope_factor: float = 32.0              # s  (Llama 3.1 RoPE scaling)
    rope_low_freq_factor: float = 1.0      # α
    rope_high_freq_factor: float = 4.0     # β
    rope_original_max_seq_len: int = 8192  # L
    max_seq_len: int = 131072
    tie_embeddings: bool = True            # lm_head = token embedding matrix

    @property
    def head_dim(self):
        return self.dim // self.n_heads


cfg = Config()

## RMSNorm

$$\mathrm{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \odot g$$

This is LayerNorm, $\frac{x - \mu}{\sigma} \odot g + \beta$, without the mean subtraction and the bias $\beta$. It needs one reduction ($\sum x_i^2$) instead of two ($\mu$, then $\sigma$).

**Scale invariance:** $\mathrm{RMSNorm}(c\,x) = \mathrm{RMSNorm}(x)$ for $c > 0$, up to $\epsilon$. The residual stream's magnitude can grow from block to block while every block still reads unit-RMS inputs.

**Cost:** $d$ parameters and about $4d$ FLOPs per token, against $2d$ elements read and written. At under 1 FLOP per byte it is memory-bound, which makes it a fusion candidate for the adjacent matmul. The sum of squares is accumulated in fp32: bf16 carries 8 bits of precision, too few for a 2048-term sum.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.type_as(x) * self.weight

## Rotary position embeddings (RoPE)

Pair the channels of a query or key as complex numbers, $z_i = x_i + \mathrm{i}\,x_{i+d_h/2}$ for $i = 0, \dots, d_h/2 - 1$. RoPE rotates pair $i$ of the vector at position $m$ by angle $m\theta_i$:

$$z_i \mapsto z_i\, e^{\mathrm{i} m\theta_i}, \qquad \theta_i = b^{-2i/d_h}$$

In real coordinates:

$$\begin{pmatrix} x'_i \\ x'_{i+d_h/2} \end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix} \begin{pmatrix} x_i \\ x_{i+d_h/2} \end{pmatrix}$$

With $q_i, k_i$ the complex pairs of $q$ and $k$, the score between a query at $m$ and a key at $n$ is

$$\langle R_m q,\, R_n k \rangle = \mathrm{Re} \sum_i q_i\, \bar{k}_i\, e^{\mathrm{i}(m-n)\theta_i}$$

It depends on position only through $m - n$: position enters attention as a relative offset, with no learned parameters. Values are not rotated, so position decides which tokens are attended to, not what is mixed.

**Frequencies** form a geometric sequence. It runs from $\theta_0 = 1$ (wavelength $2\pi \approx 6$ tokens) to $\theta_{31} = b^{-62/64}$ (wavelength $\approx 2.1$M tokens). Fast pairs resolve local order, and slow pairs vary over long distances. The base sets the longest wavelength: at $d_h = 64$, Llama 2's $b = 10^4$ gives 47K tokens and Llama 3's $b = 5 \times 10^5$ gives 2.1M.

**Llama 3.1 scaling** extends the trained context $L = 8192$ to 128K tokens. With wavelength $\lambda_i = 2\pi / \theta_i$:

$$\theta'_i = \begin{cases} \theta_i & \lambda_i < L/\beta & \text{pairs } 0\text{–}14 \\ \theta_i / s & \lambda_i > L/\alpha & \text{pairs } 18\text{–}31 \\ (1-\gamma)\,\theta_i / s + \gamma\,\theta_i & \text{otherwise} & \text{pairs } 15\text{–}17 \end{cases} \qquad \gamma = \frac{L/\lambda_i - \alpha}{\beta - \alpha}$$

Pairs with $\lambda_i < L/\beta = 2048$ complete at least 4 full rotations within the trained context, so every angle they produce at 128K was already seen in training. Pairs with $\lambda_i > L$ never completed a rotation in training. Dividing their frequency by $s = 32$ maps positions up to 128K onto angles reached by position $128\text{K}/32 = 4096 < L$.

**Cost:** no parameters. Each pair takes 6 FLOPs, so a token costs $3 d_h (H + H_{kv}) = 7680$ FLOPs per layer.

In [ ]:
def rope_cache(cfg, seq_len):
    D = cfg.head_dim
    freq = 1.0 / cfg.rope_theta ** (torch.arange(0, D, 2).float() / D)  # θ_i, (D/2,)

    wavelen = 2 * math.pi / freq                                         # λ_i
    orig = cfg.rope_original_max_seq_len
    smooth = (orig / wavelen - cfg.rope_low_freq_factor) / (cfg.rope_high_freq_factor - cfg.rope_low_freq_factor)  # γ
    freq = torch.where(
        wavelen < orig / cfg.rope_high_freq_factor, freq,                       # high freq: keep
        torch.where(
            wavelen > orig / cfg.rope_low_freq_factor, freq / cfg.rope_factor,  # low freq: stretch
            (1 - smooth) * freq / cfg.rope_factor + smooth * freq,              # in between: blend
        ),
    )

    angles = torch.outer(torch.arange(seq_len).float(), freq)                  # m·θ_i, (T, D/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x: (..., T, D); cos/sin: (T, D/2)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)


# <R_m q, R_n k> depends only on n - m
cos, sin = rope_cache(cfg, 64)
q, k = torch.randn(cfg.head_dim), torch.randn(cfg.head_dim)
rot = lambda v, p: apply_rope(v[None], cos[p : p + 1], sin[p : p + 1])[0]
torch.testing.assert_close(rot(q, 3) @ rot(k, 1), rot(q, 40) @ rot(k, 38), rtol=1e-5, atol=1e-5)

## Attention with GQA and a KV cache

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_h}} + M\right) V, \qquad M_{ij} = \begin{cases} 0 & j \le p + i \\ -\infty & \text{otherwise} \end{cases}$$

$p$ is the position of the first query. One mask covers both prefill ($p = 0$, the whole prompt as queries) and decode (a single query at position $p$).

**The $1/\sqrt{d_h}$ scale:** for $q, k$ with independent unit-variance entries, $\mathrm{Var}(q \cdot k) = d_h$. Dividing by $\sqrt{d_h}$ keeps scores at unit variance, so the softmax neither saturates to one-hot nor flattens as $d_h$ changes.

**Grouped-query attention:** query head $h$ reads K/V head $\lfloor h/g \rfloor$.

| | K/V heads | KV cache per token (bf16) |
|---|---|---|
| Multi-head (MHA) | $H = 32$ | 128 KiB |
| Grouped-query (GQA) | $H_{kv} = 8$ | 32 KiB |
| Multi-query (MQA) | 1 | 4 KiB |

Sharing K/V heads does not reduce attention FLOPs, only the bytes of $K$ and $V$. Decode attention reads the whole cache for one query per head, so its arithmetic intensity rises with the group size $g$:

$$\frac{\text{FLOPs}}{\text{bytes}} = \frac{\overbrace{2 H S d_h}^{QK^\top} + \overbrace{2 H S d_h}^{AV}}{\underbrace{2 \cdot S H_{kv} d_h \cdot 2}_{K,\,V \text{ in bf16}}} = \frac{H}{H_{kv}} = g$$

That is 1 FLOP/byte for MHA and 4 for this model.

**KV cache:** under causal masking, the keys and values at position $j$ depend only on tokens $\le j$, so appending a token never changes them. Step $t$ computes $q, k, v$ for token $t$ alone and reuses the stored $K, V$ for positions $< t$. Generating $T$ tokens costs $O(T)$ K/V projections with the cache and $O(T^2)$ without it.

**Cost per token per layer:**
- projections: $2(2d^2 + 2dH_{kv}d_h) \approx 21$M FLOPs
- scores and weighted sum over a context of $S$ tokens: $4 H d_h S = 4dS$ FLOPs

In [ ]:
class KVCache:
    """K/V buffers for one layer: (B, n_kv_heads, max_len, head_dim)."""

    def __init__(self, cfg, batch_size, max_len, device, dtype):
        shape = (batch_size, cfg.n_kv_heads, max_len, cfg.head_dim)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)

    def update(self, k, v, start_pos):
        end = start_pos + k.shape[2]
        self.k[:, :, start_pos:end] = k
        self.v[:, :, start_pos:end] = v
        return self.k[:, :, :end], self.v[:, :, :end]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.n_kv_heads, self.head_dim = cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.wq = nn.Linear(cfg.dim, cfg.n_heads * cfg.head_dim, bias=False)
        self.wk = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wv = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wo = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.dim, bias=False)

    def forward(self, x, cos, sin, cache=None, start_pos=0):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # (B, H, T, D)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # (B, Hkv, T, D)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v, start_pos)  # (B, Hkv, S, D) with S = start_pos + T

        rep = self.n_heads // self.n_kv_heads
        k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, H, T, S)
        mask = torch.ones(T, k.shape[2], dtype=torch.bool, device=x.device).tril(diagonal=start_pos)
        scores = scores.masked_fill(~mask, float("-inf"))
        out = F.softmax(scores.float(), dim=-1).type_as(q) @ v        # (B, H, T, D)
        return self.wo(out.transpose(1, 2).reshape(B, T, -1))

## SwiGLU feed-forward

$$\mathrm{FFN}(x) = W_2\big(\mathrm{SiLU}(W_1 x) \odot W_3 x\big), \qquad \mathrm{SiLU}(z) = z\,\sigma(z)$$

$W_1$ (gate) and $W_3$ (up) map $d \to d_{ff}$, and $W_2$ (down) maps back. Each hidden unit is the product of two learned projections of $x$, one passed through SiLU. A plain MLP instead passes a single projection through a fixed nonlinearity.

**Cost:** $3 d\, d_{ff} = 12d^2 \approx 50.3$M parameters per layer against $\approx 10.5$M for attention, so the FFN holds 83% of each block's weights. It does $6 d\, d_{ff} \approx 101$M FLOPs per token. A two-matrix MLP of width $4d$ has $8d^2$ parameters, and a parameter-matched SwiGLU would use $d_{ff} = \tfrac{8}{3}d$; this model goes wider.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)  # gate
        self.w3 = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)  # up
        self.w2 = nn.Linear(cfg.ffn_dim, cfg.dim, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## The model

$N$ pre-norm residual blocks:

$$h = x + \mathrm{Attn}(\mathrm{RMSNorm}(x)), \qquad x' = h + \mathrm{FFN}(\mathrm{RMSNorm}(h))$$

$$\text{logits} = \mathrm{RMSNorm}(x_N)\, E^\top, \qquad \text{logit}_v = \langle \mathrm{RMSNorm}(x_N),\, e_v \rangle$$

**Residual stream:** every block reads a normalized copy of $x$ and adds its output back. Pre-norm keeps an identity path from embedding to output, $x_N = x_0 + \sum_\ell f_\ell(\cdot)$. Post-norm, $x' = \mathrm{Norm}(x + f(x))$, renormalizes that path at every block.

**Tied embeddings:** the same $E \in \mathbb{R}^{V \times d}$ maps tokens to vectors and vectors to logits. A token's logit is the similarity of the final state to its embedding $e_v$. An untied output matrix would add $Vd \approx 263$M parameters; even tied, $E$ is 21% of the model.

**Parameters per block:** attention $2dHd_h + 2dH_{kv}d_h \approx 10.5$M, FFN $3d\,d_{ff} \approx 50.3$M, norms $2d$.

In [ ]:
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn = FeedForward(cfg)

    def forward(self, x, cos, sin, cache=None, start_pos=0):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache, start_pos)
        return x + self.ffn(self.ffn_norm(x))


class Llama(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(cfg, cfg.max_seq_len)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def make_caches(self, batch_size, max_len):
        w = self.tok_emb.weight
        return [KVCache(self.cfg, batch_size, max_len, w.device, w.dtype) for _ in self.layers]

    def forward(self, tokens, caches=None, start_pos=0):
        T = tokens.shape[1]
        cos, sin = self.cos[start_pos : start_pos + T], self.sin[start_pos : start_pos + T]
        x = self.tok_emb(tokens)
        for i, layer in enumerate(self.layers):
            x = layer(x, cos, sin, caches[i] if caches else None, start_pos)
        return self.lm_head(self.norm(x))


with torch.device(device):
    model = Llama(cfg).eval()

## Generation: prefill, then decode

Prefill runs the prompt's $T$ tokens through one forward pass and fills the cache. Each decode step then runs one token per sequence.

A weight matrix $W \in \mathbb{R}^{d_{out} \times d_{in}}$ in bf16, applied to $n$ tokens, does $2 n\, d_{in} d_{out}$ FLOPs and reads $2\, d_{in} d_{out}$ bytes of weights:

$$\text{arithmetic intensity} \approx \frac{2\, n\, d_{in} d_{out}}{2\, d_{in} d_{out}} = n \ \text{FLOPs/byte}$$

Here $n = BT$ in prefill and $n = B$ in decode. A GPU is compute-bound above its ridge point, peak FLOP/s ÷ memory bandwidth, which is 100–300 FLOPs/byte for bf16 on recent GPUs. Prefill of a few hundred tokens is compute-bound. Decode at small batch is memory-bound: reading the weights limits it.

Greedy decoding: $x_{t+1} = \arg\max_v\, p(v \mid x_{\le t})$.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens, stop_ids=()):
    B, T = prompt.shape
    caches = model.make_caches(B, T + max_new_tokens)
    stop_ids = torch.tensor(list(stop_ids), dtype=torch.long, device=prompt.device)
    done = torch.zeros(B, 1, dtype=torch.bool, device=prompt.device)

    logits = model(prompt, caches, start_pos=0)                    # prefill
    out = [logits[:, -1].argmax(-1, keepdim=True)]
    for pos in range(T, T + max_new_tokens - 1):                   # decode
        done |= torch.isin(out[-1], stop_ids)
        if done.all():
            break
        logits = model(out[-1], caches, start_pos=pos)
        out.append(logits[:, -1].argmax(-1, keepdim=True))
    return torch.cat([prompt, *out], dim=1)


prompt = torch.randint(0, cfg.vocab_size, (2, 16), device=device)
seq = generate(model, prompt, max_new_tokens=32)
seq.shape

Causal masking makes the cache exact: logits from prefill plus single-token decode steps equal the logits of one forward pass over the full sequence.

In [ ]:
with torch.no_grad():
    full = model(seq)
    caches = model.make_caches(seq.shape[0], seq.shape[1])
    P = prompt.shape[1]
    stepped = [model(seq[:, :P], caches, 0)]
    stepped += [model(seq[:, p : p + 1], caches, p) for p in range(P, seq.shape[1])]

torch.testing.assert_close(torch.cat(stepped, dim=1), full, rtol=1e-3, atol=1e-3)
del full, stepped, caches

## Back-of-envelope

$$P = N\big(\underbrace{2 d H d_h + 2 d H_{kv} d_h}_{\text{attention}} + \underbrace{3 d\, d_{ff}}_{\text{FFN}} + \underbrace{2d}_{\text{norms}}\big) + \underbrace{V d}_{\text{embedding}} + \underbrace{d}_{\text{final norm}}$$

$$\text{KV bytes per token} = 2 \cdot N \cdot H_{kv} \cdot d_h \cdot \text{bytes per element}$$

$$\text{FLOPs per token} \approx \underbrace{2P}_{\text{matmuls}} + \underbrace{4 N d S}_{\text{attention over } S \text{ tokens}}$$

Each weight takes part in one multiply-add per token. The attention term overtakes the matmuls at a context of $S = 2P / 4Nd \approx 19$K tokens.

In [ ]:
def n_params(cfg):
    attn = 2 * cfg.dim * cfg.n_heads * cfg.head_dim + 2 * cfg.dim * cfg.n_kv_heads * cfg.head_dim
    ffn = 3 * cfg.dim * cfg.ffn_dim
    per_layer = attn + ffn + 2 * cfg.dim                                 # + two RMSNorm weights
    embed = cfg.vocab_size * cfg.dim * (1 if cfg.tie_embeddings else 2)  # + lm_head unless tied
    return cfg.n_layers * per_layer + embed + cfg.dim                    # + final norm


def kv_bytes_per_token(cfg, bytes_per_elem=2):
    return 2 * cfg.n_layers * cfg.n_kv_heads * cfg.head_dim * bytes_per_elem  # K and V, every layer


assert n_params(cfg) == sum(p.numel() for p in model.parameters())

params, kv = n_params(cfg), kv_bytes_per_token(cfg)
attn_flops_per_ctx = 4 * cfg.n_layers * cfg.dim
print(f"params:           {params / 1e9:.2f} B  ({params * 2 / 2**30:.2f} GiB in bf16)")
print(f"KV cache / token: {kv / 2**10:.0f} KiB (bf16)")
print(f"KV cache @ {cfg.max_seq_len} tokens: {kv * cfg.max_seq_len / 2**30:.1f} GiB per sequence")
print(f"FLOPs / token:    {2 * params / 1e9:.2f} GFLOP + {attn_flops_per_ctx} per context token")
print(f"attention FLOPs = matmul FLOPs at context {2 * params / attn_flops_per_ctx:,.0f}")
print(f"decode ceiling at 400 GB/s, batch 1: {400e9 / (params * 2):.0f} tokens/s")

At batch size $B$ and context $S$, a decode step reads every weight once plus each sequence's KV cache:

$$t_{\text{step}} \approx \frac{2P + B \cdot S \cdot \text{KV bytes per token}}{\text{memory bandwidth}}, \qquad \text{tokens/s} = \frac{B}{t_{\text{step}}}$$

At $B = 1$ the weights dominate, giving tokens/s $\le$ bandwidth ÷ 2.47 GB. Larger batches share the weight read until the KV term catches up: at $S = 4096$, the caches of 18 sequences weigh as much as the model.

## Pretrained weights

The checkpoint stores the same $P$ parameters in bf16, $2P \approx 2.47$ GB, under Hugging Face's tensor names.

**RoPE pairing:** Meta's original checkpoints rotate interleaved pairs $(x_{2i}, x_{2i+1})$, while `apply_rope` rotates $(x_i, x_{i+d_h/2})$. A score $q \cdot k$ is unchanged when the same permutation is applied to the channels of both $q$ and $k$. Permuting the output rows of $W_q$ and $W_k$ within each head therefore converts one convention into the other, and the Hugging Face checkpoints have that permutation applied.

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from tokenizers import Tokenizer

REPO = "meta-llama/Llama-3.2-1B-Instruct"

RENAMES = [
    ("model.", ""), ("embed_tokens", "tok_emb"),
    ("self_attn.q_proj", "attn.wq"), ("self_attn.k_proj", "attn.wk"),
    ("self_attn.v_proj", "attn.wv"), ("self_attn.o_proj", "attn.wo"),
    ("mlp.gate_proj", "ffn.w1"), ("mlp.up_proj", "ffn.w3"), ("mlp.down_proj", "ffn.w2"),
    ("input_layernorm", "attn_norm"), ("post_attention_layernorm", "ffn_norm"),
]


def rename(name):
    for hf, ours in RENAMES:
        name = name.replace(hf, ours)
    return name


state = {rename(k): v for k, v in load_file(hf_hub_download(REPO, "model.safetensors")).items()}
state.setdefault("lm_head.weight", state["tok_emb.weight"])  # tied embeddings: lm_head = E

dtype = torch.bfloat16 if device == "cuda" else torch.float32
model.to(dtype).load_state_dict(state)
del state

tokenizer = Tokenizer.from_file(hf_hub_download(REPO, "tokenizer.json"))
STOP_IDS = [tokenizer.token_to_id(t) for t in ("<|eot_id|>", "<|eom_id|>", "<|end_of_text|>")]

## Chat format

```
turn(role, content) = <|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>
prompt              = <|begin_of_text|> + turn(system, …) + turn(user, …) + <|start_header_id|>assistant<|end_header_id|>\n\n
```

The prompt ends with an open assistant header. The model writes the assistant turn and closes it with `<|eot_id|>`.

In [ ]:
def chat(user_msg, system_msg=None, max_new_tokens=256):
    text = "<|begin_of_text|>"
    if system_msg:
        text += f"<|start_header_id|>system<|end_header_id|>\n\n{system_msg}<|eot_id|>"
    text += f"<|start_header_id|>user<|end_header_id|>\n\n{user_msg}<|eot_id|>"
    text += "<|start_header_id|>assistant<|end_header_id|>\n\n"

    ids = tokenizer.encode(text, add_special_tokens=False).ids
    prompt = torch.tensor([ids], device=device)
    t0 = time.perf_counter()
    reply = generate(model, prompt, max_new_tokens, STOP_IDS)[0, len(ids) :].tolist()
    dt = time.perf_counter() - t0
    print(tokenizer.decode(reply, skip_special_tokens=True))
    print(f"\n[{len(reply)} tokens, {len(reply) / dt:.1f} tokens/s]")


chat("Explain what a KV cache is in two sentences.")

In [ ]:
chat("Write a haiku about GPUs.")